In [ ]:
# ==============================
# 
# GSLC Download script- Downloads and crops all available NISAR GSLC's within the user defined region and accord to defined filters: 
#   date, polarization (HH,HV), look direction (A,D), frequency (A,B)
#
#   IMPORTANT NOTE: Certain filters may not be fully functional yet, specifically polarization and look direction
#                   Leaving those filters as None will download all available polarizations and look directions within the other filters and limits
#
# 
# ==============================
from osgeo import gdal, osr
import numpy as np
import datetime
import netrc
from shapely.geometry import box, MultiPolygon
import asf_search
import asf_search.constants.INTERNAL as asf_internal
import earthaccess
from tqdm import tqdm
from pathlib import Path

asf_internal.CMR_TIMEOUT = 120

# ============================
# AUTHENTICATION (via ~/.netrc)
# ============================
# Requires a ~/.netrc entry:
#   machine urs.earthdata.nasa.gov
#   login your_username
#   password your_password
# chmod 600 ~/.netrc

auth = earthaccess.login(strategy='netrc')
endpoint = 'https://nisar.asf.earthdatacloud.nasa.gov/s3credentials'
s3_credentials = auth.get_s3_credentials(endpoint=endpoint)

netrc_login, _, netrc_password = netrc.netrc().authenticators('urs.earthdata.nasa.gov')

gdal.SetConfigOption('GDAL_HTTP_AUTH', None)
gdal.SetConfigOption('GDAL_HTTP_USERPWD', None)
gdal.SetConfigOption('AWS_ACCESS_KEY_ID', None)
gdal.SetConfigOption('AWS_SECRET_ACCESS_KEY', None)
gdal.SetConfigOption('AWS_SESSION_TOKEN', None)

session = asf_search.ASFSession()
session.auth_with_creds(netrc_login, netrc_password)

# ===================================================================================

# USER CONFIGURATION- change these to reflect target area, time, and iamge filters

# ===================================================================================
lat_min = -77.7041  # southernmost latitude
lon_min = 166.335  # westernmost longitude
lat_max = -77.3687  # northernmost latitude
lon_max = 167.95  # easternmost longitude

bbox = (lon_min, lat_min, lon_max, lat_max)
start_date = "2025-08-01"
end_date   = "2026-06-01"
polarization = ['HH']
look_direction = None
frequency = 'A'
outDir = "/home/matt/NISAR_cal_eval/Erebus"  # <---- NOTE: Change this to your desired output destination!
fileType = "GSLC"
min_coverage= .8   #<---------- NOTE: Set to minimum % coverage of the bounding box needed for a granule to be downloaded

# ==================================
# Function Definitions and Functional Code
# ==================================
from shapely.geometry import box, shape, MultiPolygon

def search_nisar(bbox, fileType, session,
                  start_date=None, end_date=None,
                  polarization=None,
                  min_coverage=0.8):  # <-- NEW: fraction 0-1, e.g. 0.8 = require 80% coverage
    """
    ...
    min_coverage: float or None
        If set, only keep granules whose footprint covers at least this
        fraction of the bounding box area (0.0-1.0). None = no filtering.
    """
    aoi_poly = box(bbox[0], bbox[1], bbox[2], bbox[3])
    aoi_wkt = aoi_poly.wkt
    aoi_area = aoi_poly.area  

    asf_search.constants.INTERNAL.CMR_TIMEOUT = 120

    if polarization and isinstance(polarization, str):
        polarization = [polarization]

    opts = asf_search.ASFSearchOptions(
        collections=[
            "C2850259510-ASF",
            "C4052499921-ASF",
            "C2850261892-ASF"
        ],
        processingLevel=fileType,
        intersectsWith=aoi_wkt,
        session=session,
        dataset=asf_search.constants.NISAR
    )

    results = asf_search.geo_search(opts=opts)
    print(f"Found {len(results)} granules intersecting bounding box")

    def to_utc(dt_str):
        if dt_str is None:
            return None
        dt = datetime.datetime.fromisoformat(dt_str)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=datetime.timezone.utc)
        return dt

    start_dt = to_utc(start_date)
    end_dt   = to_utc(end_date)

    filtered = []
    for r in results:
        t = datetime.datetime.fromisoformat(r.properties["startTime"])
        if t.tzinfo is None:
            t = t.replace(tzinfo=datetime.timezone.utc)

        if start_dt and t < start_dt:
            continue
        if end_dt and t > end_dt:
            continue

        if polarization:
            pols = r.properties.get("mainBandPolarization") or []
            if isinstance(pols, str):
                pols = [pols]
            fname = r.properties.get("fileName", "")
            if not (any(p in pols for p in polarization) or
                    any(p in fname for p in polarization)):
                continue

        # ---- NEW: coverage filter ----
        if min_coverage is not None:
            try:
                footprint = shape(r.geometry)  # granule footprint as shapely geometry
            except Exception as e:
                print(f"⚠️ Could not parse geometry for {r.properties.get('fileName')}: {e}")
                continue

            intersection = footprint.intersection(aoi_poly)
            coverage = intersection.area / aoi_area if aoi_area > 0 else 0

            # stash it on the result for later inspection/debugging
            r.properties["aoi_coverage"] = coverage

            if coverage < min_coverage:
                continue
        # --------------------------------

        filtered.append(r)

    print(f"Granules after optional filters: {len(filtered)}")
    return filtered


def discover_TF(results):
    seen = {}
    for r in results:
        orbit = int(r.properties["pathNumber"])
        frame = int(r.properties["frameNumber"])
        pol   = r.properties.get("mainBandPolarization")
        if isinstance(pol, list):
            pol = pol[0] if pol else None

        key = (orbit, frame)
        if key not in seen:
            seen[key] = set()
        if pol:
            seen[key].add(pol)

    if not seen:
        print("❌ No TF combinations found")
        return None, None

    orbits = sorted(seen.keys())
    TF = np.array([[o, f] for o, f in orbits])
    poles = [seen[k] for k in orbits]   # each entry is now a SET, e.g. {'HH','HV'}
    return TF, poles


def get_SLCs_date_range(
    TF, TFpol, session, projWin, fileType, mbox, interp, frequency,
    start_date, end_date, granules, polarization,
    outDir,
):
    # ---- GDAL / VSICURL tuning ----
    gdal.SetConfigOption('CPL_VSIL_CURL_CHUNK_SIZE', '8388608')  # 8 MB chunks
    gdal.SetConfigOption('AWS_NO_SIGN_REQUEST', 'NO')
    gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN', 'EMPTY_DIR')  # skips ListBucket
    gdal.SetConfigOption('CPL_VSIL_CURL_ALLOWED_EXTENSIONS', '.h5')
    gdal.SetConfigOption('GDAL_CACHEMAX', '512')  # MB

    # Convert user inputs to timezone-aware UTC datetimes
    start_date = datetime.datetime.fromisoformat(start_date)
    end_date   = datetime.datetime.fromisoformat(end_date)

    if start_date.tzinfo is None:
        start_date = start_date.replace(tzinfo=datetime.timezone.utc)
    if end_date.tzinfo is None:
        end_date = end_date.replace(tzinfo=datetime.timezone.utc)

    pols    = polarization
    polyIds = np.zeros((len(mbox.geoms)), dtype='object')
    allData = np.zeros((len(pols)), dtype='object')

    results = granules
    print(f"Total matching public granules found: {len(results)}")

    if len(results) == 0:
        print("❌ No granules found for this orbit/frame combination.")
        return [], [], [], [], []

    # Filter by date range
    filtered = []
    for r in results:
        t = datetime.datetime.fromisoformat(r.properties['startTime'])
        if t.tzinfo is None:
            t = t.replace(tzinfo=datetime.timezone.utc)
        if start_date <= t <= end_date:
            filtered.append(r)

    print(f"Granules within date range: {len(filtered)}")

    if len(filtered) == 0:
        print("❌ No granules fall within the requested date range.")
        return [], [], [], [], []

    # Filter by polarization
    s3List     = []
    resultList = []
    polList    = []
    dates      = []
    crids      = []
    seen_files = set()

    for result in filtered:
        main_pols = result.properties.get('mainBandPolarization') or []
        side_pols = result.properties.get('sideBandPolarization') or []
        if isinstance(main_pols, str):
            main_pols = [main_pols]
        if isinstance(side_pols, str):
            side_pols = [side_pols]

        pol_match = (
            any(p in main_pols for p in polarization) or
            any(p in side_pols for p in polarization) or
            any(p in result.properties['fileName'] for p in polarization)
        )
        if not pol_match:
            continue

        file_name = result.properties['fileName']
        if file_name in seen_files:
            continue
        seen_files.add(file_name)

        s3s = result.properties.get('s3Urls') or []
        if s3s:
            s3List.append(next((s3 for s3 in s3s if s3.endswith('.h5')), s3s[0]))
        else:
            s3List.append(result.properties['url'])

        resultList.append(result)
        polList.append(result.properties['mainBandPolarization'])
        dates.append(datetime.datetime.fromisoformat(result.properties['startTime']))
        crids.append(result.properties['crid'])

    # Sort by date
    indices = np.argsort(dates)
    dates      = [dates[i]      for i in indices]
    polList    = [polList[i]    for i in indices]
    s3List     = [s3List[i]     for i in indices]
    resultList = [resultList[i] for i in indices]
    crids      = [crids[i]      for i in indices]

    print(f"Granules after polarization filter: {len(resultList)}")

    # Process remote granules
    outDir = Path(outDir)
    outDir.mkdir(parents=True, exist_ok=True)

    polStatus = np.zeros((len(resultList), len(pols)))
    starting  = np.ones(len(pols))

    geoTransforms = {}
    projections   = {}

    auth_session = auth.get_session()

    target_srs = osr.SpatialReference()
    source_srs = osr.SpatialReference()
    source_srs.ImportFromEPSG(4326)

    creds = auth.get_s3_credentials(daac='ASF')

    gdal.SetConfigOption('AWS_ACCESS_KEY_ID',     creds['accessKeyId'])
    gdal.SetConfigOption('AWS_SECRET_ACCESS_KEY', creds['secretAccessKey'])
    gdal.SetConfigOption('AWS_SESSION_TOKEN',     creds['sessionToken'])
    gdal.SetConfigOption('AWS_DEFAULT_REGION',    'us-west-2')

    print("\nProcessing granules:")
    for i, result in enumerate(tqdm(resultList, desc="Granules", unit="granule")):

        response = auth_session.get(result.properties['url'], allow_redirects=True, stream=True)
        download_url = response.url
        response.close()

        remote_path = '/vsicurl/' + download_url
        dataset = gdal.Open(remote_path, gdal.GA_ReadOnly)
        if dataset is None:
            print(f"Failed to open {remote_path}")
            continue

        subdatasets = dataset.GetSubDatasets()

        for j, pol in enumerate(pols):

            if fileType == 'GSLC':
                suffix = f'/science/LSAR/{fileType}/grids/frequency{frequency}/{pol}'
            else:
                suffix = f'/science/LSAR/{fileType}/grids/frequency{frequency}/{pol}{pol}'

            matching_ds = next((ds_info[0] for ds_info in subdatasets if suffix in ds_info[0]), None)
            if matching_ds is None:
                continue

            polStatus[i, j] = 1

            mem_ds = gdal.Translate(
                '', matching_ds,
                format='MEM',
                projWin=projWin,
                projWinSRS='EPSG:4326',
                xRes=5, yRes=5,
                resampleAlg=interp
            )

            file_stem = Path(result.properties['fileName']).stem
            
            crop_name = f"{file_stem}_{pol}_freq{frequency}.tif"
            crop_path = outDir / crop_name

            gdal.Translate(
                str(crop_path),
                mem_ds,
                format='GTiff',
                creationOptions=['COMPRESS=LZW']
            )

            tmp = mem_ds.GetRasterBand(1).ReadAsArray()

            if starting[j]:
                ny, nx = tmp.shape

                if fileType == 'GSLC':
                    data = np.zeros((len(resultList), ny, nx), dtype='complex64')
                else:
                    data = np.zeros((len(resultList), ny, nx))

                geoTransforms[j] = mem_ds.GetGeoTransform()
                projections[j]   = mem_ds.GetProjection()

                inv_gt = gdal.InvGeoTransform(geoTransforms[j])
                target_srs.ImportFromWkt(projections[j])
                ct = osr.CoordinateTransformation(source_srs, target_srs)

                for k, geom in enumerate(mbox.geoms):
                    lonlat = list(geom.exterior.coords)
                    coords = []
                    for l in range(len(lonlat)):
                        utm_x, utm_y, _ = ct.TransformPoint(lonlat[l][1], lonlat[l][0])
                        coords.append(gdal.ApplyGeoTransform(inv_gt, utm_x, utm_y))
                    polyIds[k] = coords

                starting[j] = 0
            else:
                data = allData[j]

            data[i, :, :] = tmp
            allData[j]    = data

        dataset = None

    return dates, polStatus, allData, polyIds, crids


def run_nisar_pipeline(
    bbox,
    fileType,
    session,
    outDir,
    start_date=start_date,
    end_date=end_date,
    polarization=polarization,
    frequency=frequency,
    min_coverage=0.8
):
    # 1. Search ASF
    results = search_nisar(
        bbox=bbox,
        fileType=fileType,
        session=session,
        start_date=start_date,
        end_date=end_date,
        polarization=polarization,
        min_coverage=min_coverage
    )
    if len(results) == 0:
        print("❌ No granules found after filtering")
        return

    # 2. Discover TF combinations
    TF, poles = discover_TF(results)
    if TF is None:
        print("❌ No TF combinations found")
        return

    min_lon, min_lat, max_lon, max_lat = bbox
    mbox = MultiPolygon([box(min_lon, min_lat, max_lon, max_lat)])

    # 3. Loop through TFs
    print("\nProcessing TF combinations:")
    for idx in tqdm(range(len(TF)), desc="TFs", unit="TF"):
        orbit, frame = TF[idx]
        pol = poles[idx]

        tf_results = [
            r for r in results
            if int(r.properties["pathNumber"]) == orbit
            and int(r.properties["frameNumber"]) == frame
        ]
        tf_results = sorted(tf_results, key=lambda r: r.properties["startTime"])

        print(f"\n  TF {idx+1}/{len(TF)} → Orbit {orbit}, Frame {frame}, Pol {pol}")
        print(f"  Granules in this TF: {len(tf_results)}")

        if len(tf_results) == 0:
            print("  ⚠️  Skipping — no granules for this TF")
            continue

        get_SLCs_date_range(
            TF=(orbit, frame),
            TFpol=pol,
            session=session,
            projWin=[min_lon, max_lat, max_lon, min_lat],
            fileType=fileType,
            mbox=mbox,
            interp="bilinear",
            start_date=start_date,
            end_date=end_date,
            outDir=outDir,
            granules=tf_results,
            frequency=frequency,
            polarization=polarization
        )

    print("\n🎉 Pipeline complete — all GSLC crops saved.")


run_nisar_pipeline(
    bbox=bbox,
    fileType=fileType,
    session=session,
    outDir=outDir,
    start_date=start_date,
    end_date=end_date,
    polarization=polarization,
    frequency=frequency,
    min_coverage=min_coverage
)

Found 194 granules intersecting bounding box
Granules after optional filters: 64

Processing TF combinations:


TFs:   0%|          | 0/3 [00:00<?, ?TF/s]


  TF 1/3 → Orbit 23, Frame 139, Pol {'HH'}
  Granules in this TF: 21
Total matching public granules found: 21
Granules within date range: 21
Granules after polarization filter: 21


/home/matt/miniconda3/envs/nisar_venv/lib/python3.14/site-packages/osgeo/osr.py:424: FutureWarning: Neither osr.UseExceptions() nor osr.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(



Processing granules:
